In [1]:
import sys; sys.path.append("../src")
import time, torch
from data_prep import prepare_data
from models import IDSNet
from attacks import fgsm
from train import train_model

d = prepare_data("../data/KDDTrain.txt", "../data/KDDTest.txt")
X_train = torch.tensor(d["X_train"]); y_train = torch.tensor(d["y_train"], dtype=torch.long)
X_val   = torch.tensor(d["X_val"]);   y_val   = torch.tensor(d["y_val"], dtype=torch.long)
X_test  = torch.tensor(d["X_test"]);  y_test  = torch.tensor(d["y_test"], dtype=torch.long)

In [2]:
torch.manual_seed(42)
hardened = IDSNet(X_train.shape[1])
t0 = time.time()
train_model(hardened, X_train, y_train, X_val, y_val, epochs=10, adv_eps=0.1)
print(f"training time: {time.time() - t0:.0f}s")
torch.save(hardened.state_dict(), "../saved_models/nn_hardened.pt")

epoch 1: loss=0.1250  val_acc=0.9838
epoch 2: loss=0.0493  val_acc=0.9911
epoch 3: loss=0.0401  val_acc=0.9907
epoch 4: loss=0.0349  val_acc=0.9926
epoch 5: loss=0.0318  val_acc=0.9922
epoch 6: loss=0.0300  val_acc=0.9931
epoch 7: loss=0.0280  val_acc=0.9938
epoch 8: loss=0.0269  val_acc=0.9938
epoch 9: loss=0.0262  val_acc=0.9935
epoch 10: loss=0.0249  val_acc=0.9935
training time: 16s


In [3]:
baseline = IDSNet(X_train.shape[1])
baseline.load_state_dict(torch.load("../saved_models/nn_baseline.pt"))
baseline.eval(); hardened.eval()

def report(model, X, y):
    with torch.no_grad():
        p = model(X).argmax(dim=1)
    return (p == y).float().mean().item(), (p[y == 1] == 1).float().mean().item()

print(f"{'model':>9} | {'val acc':>7} | {'test acc':>8} | {'test attack recall':>18}")
for name, m in [("baseline", baseline), ("hardened", hardened)]:
    va, _ = report(m, X_val, y_val)
    ta, tr = report(m, X_test, y_test)
    print(f"{name:>9} | {va:>7.3f} | {ta:>8.3f} | {tr:>18.3f}")

    model | val acc | test acc | test attack recall
 baseline |   0.994 |    0.780 |              0.667
 hardened |   0.994 |    0.779 |              0.643


In [4]:
def evasion_rate(model, X, y, eps):
    with torch.no_grad():
        caught = model(X).argmax(dim=1) == 1
    mask = (y == 1) & caught
    X_adv = fgsm(model, X[mask], y[mask], eps)
    with torch.no_grad():
        return (model(X_adv).argmax(dim=1) == 0).float().mean().item()

print(f"{'eps':>5} | {'baseline':>9} | {'hardened':>9}")
for eps in [0.0, 0.05, 0.1, 0.2, 0.5, 1.0]:
    b = evasion_rate(baseline, X_val, y_val, eps)
    h = evasion_rate(hardened, X_val, y_val, eps)
    print(f"{eps:>5.2f} | {b*100:>8.1f}% | {h*100:>8.1f}%")

  eps |  baseline |  hardened
 0.00 |      0.0% |      0.0%
 0.05 |     27.8% |      0.1%
 0.10 |     97.1% |      0.5%
 0.20 |     99.6% |      3.4%
 0.50 |    100.0% |     92.9%
 1.00 |     99.9% |     97.9%
